In [9]:
from pymodbus.client import ModbusSerialClient as ModbusClient
import struct
import time
import datetime
import argparse


In [10]:
probe = ModbusClient(port="COM21", timeout=1, baudrate=19200, stopbits=2, parity='N')

In [14]:
def holding_registers_data():
    try:        
        registers = probe.read_holding_registers(address=0,count=10, unit=240).registers
    except Exception as e:
        print(e)
        return False, None, None, None
    try:
        rh = data_from_register(registers, 1)
        t = data_from_register(registers, 3)  
        dp = data_from_register(registers,9)   
       
    except Exception as e:
        print(e)
        return False, None, None, None
        return True, rh, t, dp

In [18]:
from pymodbus.client import ModbusSerialClient as ModbusClient
import struct
import time
import datetime
import argparse

port = "COM21"
adress = 240
hours = 1
rate = 1

parser = argparse.ArgumentParser(
    description="Modbus data logger"
)

probe = ModbusClient(port=port, timeout=1, baudrate=19200, stopbits=2, parity='N')


end = datetime.datetime.now() + datetime.timedelta(hours=hours)

print("End date and time: ", end)


#Converts the data from two registers to a 32-bit float
#Takes the holding register data and the index of the register as inputs
#Returns 32-bit float
def data_from_register(registers, i):
    return struct.unpack('!f', bytes.fromhex('{0:04x}'.format(registers[i]) + '{0:04x}'.format(registers[i-1])))[0]


#Reads the holding registers of the probe and returns the values as 32-bit float
#Returns True, Relative Humidity, Temperature and Dew Point if read sucessfully
#Returns False, None, None, None if not
def holding_registers_data():
    try:
        
        registers = probe.read_holding_registers(address=0,count=10).registers
        
        
    except Exception as e:
        print(e)
        return False, None, None, None
    try:
        rh = data_from_register(registers, 1)
        t = data_from_register(registers, 3)  
        dp = data_from_register(registers,9)   
       
    except Exception as e:
        print(e)
        return False, None, None, None
    
    return True, rh, t, dp


#Reads relative humidity, temperature and dew point from holding_registers_data() and writes the values to a csv file with the date and time
def data_logger():
    probe.connect()
    successful, rh, t, dp = holding_registers_data()
    if (successful):
        dt = datetime.datetime.now()
        
        try:
            with open("data.csv", "a") as f:
                line = f"{dt},{rh},{t},{dp}\n"
                print(line)
                f.write(line)
        except Exception as e:
            print(e)
        probe.close()
        time.sleep(rate)
        
    else:
        probe.close()
        time.sleep(0.5)
    
  

def main():
    while datetime.datetime.now() < end:
        data_logger()


if __name__ == "__main__":
    main()

End date and time:  2026-08-13 14:46:24.377100


No response received after 3 retries, continue with next request


Modbus Error: [Input/Output] No response received after 3 retries, continue with next request


KeyboardInterrupt: 

In [2]:
#!/usr/bin/env python3
"""Read RH, temperature, dew point from a Vaisala HMP110 (Modbus RTU)."""

import struct
import sys
import time

from pymodbus.client import ModbusSerialClient

PORT = "COM21"   # e.g. "COM6" on Windows
SLAVE_ADDR = 240        # default HMP110 Modbus address
POLL_INTERVAL = 1.0     # seconds


def regs_to_float(registers: list[int], hi_idx: int) -> float:
    """HMP110 stores 32-bit floats as two 16-bit regs, little-endian word order."""
    raw = registers[hi_idx].to_bytes(2, "big") + registers[hi_idx - 1].to_bytes(2, "big")
    return struct.unpack("!f", raw)[0]


import inspect

def read_holding_registers_compat(client, address, count, slave):
    """Call read_holding_registers regardless of pymodbus's kwarg naming
    for the slave/unit id, which has changed across versions (unit -> slave
    -> device_id)."""
    sig = inspect.signature(client.read_holding_registers)
    params = sig.parameters
    kwargs = {"address": address, "count": count}
    for name in ("slave", "unit", "device_id"):
        if name in params:
            kwargs[name] = slave
            break
    return client.read_holding_registers(**kwargs)


def read_measurements(client: ModbusSerialClient):
    rr = read_holding_registers_compat(client, 0, 10, SLAVE_ADDR)
    if rr.isError():
        raise IOError(rr)
    regs = rr.registers
    rh = regs_to_float(regs, 1)   # relative humidity [%RH]
    t = regs_to_float(regs, 3)    # temperature [°C]
    dp = regs_to_float(regs, 9)   # dew point [°C]
    return rh, t, dp


def main():
    client = ModbusSerialClient(
        port=PORT,
        baudrate=19200,
        bytesize=8,
        parity="N",
        stopbits=2,
        timeout=1,
    )

    if not client.connect():
        sys.exit(f"Could not open {PORT}")

    try:
        while True:
            try:
                rh, t, dp = read_measurements(client)
                print(f"RH: {rh:6.2f} %RH   T: {t:6.2f} degC   Td: {dp:6.2f} degC")
            except IOError as e:
                print(f"Read error: {e}")
            time.sleep(POLL_INTERVAL)
    except KeyboardInterrupt:
        pass
    finally:
        client.close()


if __name__ == "__main__":
    main()

RH:  16.54 %RH   T:  23.35 degC   Td:  -3.04 degC
RH:  16.54 %RH   T:  23.35 degC   Td:  -3.04 degC
RH:  16.54 %RH   T:  23.35 degC   Td:  -3.04 degC
RH:  16.53 %RH   T:  23.35 degC   Td:  -3.04 degC
RH:  16.53 %RH   T:  23.35 degC   Td:  -3.04 degC
RH:  16.52 %RH   T:  23.35 degC   Td:  -3.04 degC
RH:  16.52 %RH   T:  23.35 degC   Td:  -3.05 degC
RH:  16.52 %RH   T:  23.35 degC   Td:  -3.05 degC
RH:  16.51 %RH   T:  23.35 degC   Td:  -3.06 degC
RH:  16.49 %RH   T:  23.35 degC   Td:  -3.06 degC
RH:  16.47 %RH   T:  23.35 degC   Td:  -3.08 degC
RH:  16.50 %RH   T:  23.35 degC   Td:  -3.06 degC
RH:  17.74 %RH   T:  23.35 degC   Td:  -2.20 degC
RH:  17.74 %RH   T:  23.35 degC   Td:  -2.20 degC
RH:  20.46 %RH   T:  23.35 degC   Td:  -0.49 degC
RH:  24.52 %RH   T:  23.35 degC   Td:   1.97 degC
RH:  28.72 %RH   T:  23.35 degC   Td:   4.19 degC
RH:  32.36 %RH   T:  23.35 degC   Td:   5.90 degC
RH:  32.36 %RH   T:  23.35 degC   Td:   5.90 degC
RH:  35.32 %RH   T:  23.35 degC   Td:   7.17 degC
